# Chess-Coach Agent — Gemma 4 E4B QLoRA on Kaggle (2×T4) — FINAL RUN

The aimed final run. Everything learned across v1→v3 is locked in. After this, the model is
frozen and only the harness is hardened.

## Just run it
1. Settings → Accelerator → **GPU T4 ×2**.  2. Internet **On**.
3. Add-ons → Secrets → **HF_TOKEN** (WRITE scope — base download + checkpoint push).
4. Accept the Gemma 4 license once.
5. **Run top → bottom.** Cell 1 is pre-filled — no edits. Restart if Cell 4 shows a banner.

## What's locked in
- **all-linear + FORMAT_WEIGHT=8.0** → fixed the format collision (v3: train-row OK 6/6, STRESS 10/10).
- **FORMAT_WEIGHT now also covers the skill/tool NAME** → fixes the residual garbled names
  (`hood-human-chat`); names trained at weight 1.0 before.
- **LR 1e-4, 1000 steps** → v3 was overfit-free with val still DESCENDING at 600 (undertrained);
  more steps let val bottom out and names/content firm up. ~1.5–2 sessions via auto-resume.

## The proof gate (Cell 6.6) — run it, ~25 min, before the 11h train
Hard-overfits 256 rows with the new loss and free-gen probes them. **NAMES must now be exact
(`chess-coach`, not `hood-human-chat`).** If the gate passes → run Cell 7. If not → stop and
tell Opus; don't burn the session.

## Watch during train (two signals)
- Clean `<skill>`/`<tool>` with the RIGHT name by ~step 100.
- `val_loss` (every 100) dropping; `best/` saves the lowest and auto-mirrors to CKPT_REPO.

## Stop rule (don't over-spend)
After any session, run the serve notebook's probe on `ckpt-v4/best`. Ship when: train-row
OK ≥ 5 with correct names, STRESS ≥ 9/10, unseen-skills ≥ 3/3. You need not finish all 1000.

## OOM ladder (never cut seq — it's the data floor)
RANK 16→8 (keep all-linear) → DDP=False (1 GPU). seq stays 1664.


In [ ]:
# Cell 1 — config (E4B QLoRA, Kaggle 2×T4 DDP). FINAL aimed run. Pre-filled, run top-to-bottom.
import os

MODEL  = "gemma4_e4b"
ENGINE = "unsloth"
HF_REPO = {
    ("gemma4_e4b", "unsloth"): "unsloth/gemma-4-E4B-it",
    ("gemma4_e2b", "unsloth"): "unsloth/gemma-4-E2B-it",
    ("gemma4_e4b", "cuda"):    "google/gemma-4-E4B-it",
    ("gemma4_e2b", "cuda"):    "google/gemma-4-E2B-it",
}[(MODEL, ENGINE)]
NO_4BIT = False

CHESS_REPO_URL = "https://github.com/RyanDev1st/llm-and-engine.git"
CHESS_BRANCH = "feat/chess-coach-sft"
WORKDIR = "/kaggle/working/llm-and-engine"
OUTPUT = "gemma4_chess_e4b_kaggle"
DATA_STEM = "v1_2"

MAX_SEQ = 1664
TARGETS = "all-linear"   # required for format emission (proven)
RANK = 16                # proven sufficient; rank 32 only worsens overfit — do NOT bump
GRAD_ACCUM = 16
DDP = True

# FINAL run — everything proven across v1->v3 locked in:
#  - all-linear + FORMAT_WEIGHT=8.0 fixed the format collision (v3 train-row OK 6/6, STRESS 10/10).
#  - FORMAT_WEIGHT now ALSO covers the skill/tool NAME (data_pipeline) -> fixes the residual
#    garbled names (`hood-human-chat`); the name trained at weight 1.0 before.
#  - serve-side rep-penalty is OFF (it was corrupting name copying); this also LOCKS names in
#    the weights so they don't depend on the decode setting.
# LR 1e-4 (v3 was overfit-free, val monotone DOWN). MORE steps than v3's 600 — v3 val was
# STILL DESCENDING at 600 = undertrained. 1000 steps (~1.5-2 sessions via auto-resume) lets
# val bottom out and names/content firm up. Skill-vs-tool VERB bias is the HARNESS's job
# (deterministic catalog coercion), NOT a corpus change.
LR = 1e-4
MAX_STEPS = 1000
EVAL_EVERY = 100
MAX_VAL = 128
SAVE_EVERY = 50

# FRESH v4 repo — the loss changed again (name-weighting); do NOT reuse v3/v2/v1 checkpoints.
CKPT_REPO = "RyanDev1st/gemma4-chesscoach-ckpt-v4"
RESUME = False          # auto-managed by Cell 4.5
RESUME_DATASET = ""
if CKPT_REPO:
    os.environ["CHESS_CKPT_REPO"] = CKPT_REPO

print("config:", MODEL, ENGINE, "| seq", MAX_SEQ, "| targets", TARGETS, "| rank", RANK,
      "| lr", LR, "| steps", MAX_STEPS, "| ckpt", CKPT_REPO or "(local only)")


In [ ]:
# Cell 2 — GPU preflight. If this fails: Settings (right panel) → Accelerator → GPU T4 ×2,
# then re-run. A CPU kernel has no `nvidia-smi` and cannot train E4B.
import subprocess, shutil, torch
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                          "--format=csv"], capture_output=True, text=True).stdout)
else:
    print("nvidia-smi NOT found — no GPU attached to this kernel.")
assert torch.cuda.is_available(), (
    "No GPU — Settings → Accelerator → GPU T4 ×2 (×2 needed for DDP), then re-run. "
    "Batch/commit runs default to CPU; you MUST select the accelerator before running.")
n = torch.cuda.device_count()
print("cuda", torch.version.cuda, "| device", torch.cuda.get_device_name(0), "| count", n)
if n < 2:
    print("\nℹ️  Only 1 GPU — set DDP=False in Cell 1, or pick GPU T4 ×2 for the ~2x DDP speedup.")


In [ ]:
# Cell 3 — clone repo (skip LFS weights; we only need code + data)
import os, subprocess

def run(cmd, **kw):
    print(">", " ".join(cmd)); return subprocess.run(cmd, check=True, text=True, **kw)

env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
if not os.path.exists(WORKDIR):
    run(["git", "clone", "--depth", "1", "--branch", CHESS_BRANCH, CHESS_REPO_URL, WORKDIR], env=env)
else:
    run(["git", "-C", WORKDIR, "pull", "--ff-only"], env=env)
run(["git", "-C", WORKDIR, "log", "-1", "--oneline"])
os.environ["PYTHONPATH"] = f"{WORKDIR}/src/llm"
print("PYTHONPATH=", os.environ["PYTHONPATH"])


In [ ]:
# Cell 4 — deps. Let Unsloth resolve its own current stack (do NOT pin an old
# transformers — Gemma 4 needs a recent one).
import subprocess, sys
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)

if ENGINE == "unsloth":
    pip("--upgrade", "unsloth", "unsloth_zoo", "bitsandbytes")
    pip("python-chess")
    print("unsloth installed. If Kaggle shows a RESTART banner, restart, then run Cells 5+. "
          "If Gemma 4 fails to load with a model-type error, run: pip install -U transformers")
else:
    pip("-U", "transformers", "accelerate", "peft", "trl",
        "bitsandbytes", "datasets", "sentencepiece", "protobuf", "python-chess")
    import transformers, peft, bitsandbytes
    print("transformers", transformers.__version__, "| peft", peft.__version__,
          "| bnb", bitsandbytes.__version__)


In [ ]:
# Cell 4.5 — HF login + AUTO-RESUME pull + CONFIRM (runs BEFORE the base-model download
# and fit tests). Catches a bad/empty checkpoint in ~30s instead of after minutes of
# weight loading. Pulls checkpoint/ (latest, resume) + best/ (lowest-val, serve) from
# CKPT_REPO and SETS `RESUME` from what actually lands on disk. Re-run the notebook each
# session; this continues automatically — no flag, no download, no Kaggle Dataset.
import os, glob, shutil
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))   # also authenticates Cell 5 base download

dst = f"{WORKDIR}/runs/{OUTPUT}"
os.makedirs(dst, exist_ok=True)

def _has_state(d):
    return os.path.exists(f"{d}/checkpoint/trainer_state.pt")

if CKPT_REPO:
    from huggingface_hub import snapshot_download
    try:
        snapshot_download(repo_id=CKPT_REPO, repo_type="model",
                          allow_patterns=["checkpoint/*", "best/*"], local_dir=dst)
        print("Hub pull OK:", CKPT_REPO)
    except Exception as e:
        print("Hub pull skipped (", repr(e), ") — fresh start unless a local checkpoint exists")
elif RESUME:
    # Legacy fallback: stage from a Kaggle Dataset (zip or folder) in RESUME_DATASET.
    src = RESUME_DATASET
    zips = glob.glob(f"{src}/**/*.zip", recursive=True)
    if zips:
        print("unzip", zips[0], "->", dst); shutil.unpack_archive(zips[0], dst)
    elif os.path.isdir(src):
        for sub in ("checkpoint", "best"):
            s = os.path.join(src, sub)
            if not os.path.isdir(s):
                s = next((q for q in glob.glob(f"{src}/**/{sub}", recursive=True)), None)
            if s and os.path.isdir(s):
                shutil.copytree(s, os.path.join(dst, sub), dirs_exist_ok=True)
                print("copied", s, "->", os.path.join(dst, sub))

# CONFIRM before any weight loading: resume only if BOTH the trainer state AND the adapter
# weights are present (exactly what train_unsloth._load_ckpt needs). Report the step.
RESUME = False
ck = f"{dst}/checkpoint"
adapter = os.path.exists(f"{ck}/adapter_model.safetensors") or os.path.exists(f"{ck}/adapter_model.bin")
if _has_state(dst) and adapter:
    import torch
    st = torch.load(f"{ck}/trainer_state.pt", map_location="cpu")
    bv = st.get("best_val"); bv = f"{bv:.4f}" if isinstance(bv, (int, float)) else bv
    RESUME = True
    print(f"✅ CONFIRMED resume @ update {st.get('update')}/{MAX_STEPS} "
          f"(epoch {st.get('epoch')}, best_val {bv}). Safe to load weights / train.")
    print("   checkpoint files:", os.listdir(ck))
    if os.path.isdir(f"{dst}/best"):
        print("   best/ present (serve):", os.listdir(f"{dst}/best"))
elif _has_state(dst) and not adapter:
    raise SystemExit("checkpoint/trainer_state.pt present but adapter weights MISSING — bad "
                     "pull. Fix CKPT_REPO before wasting time on the base-model download.")
else:
    print("No checkpoint — FRESH run (first session). RESUME=False.")


In [ ]:
# Cell 5 — HF login + download base model (token from Kaggle Secrets)
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download

login(UserSecretsClient().get_secret("HF_TOKEN"))
dest = f"{WORKDIR}/src/llm/models/{MODEL}"
snapshot_download(repo_id=HF_REPO, local_dir=dest,
                  allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "tokenizer*"])
print("base model at", dest)
print(sorted(os.listdir(dest)))


In [ ]:
# Cell 6 - data check (must be the REGENERATED grounded corpus; stored gzipped)
import os, gzip
for name in (f"{DATA_STEM}_train.jsonl", f"{DATA_STEM}_val.jsonl"):
    base = f"{WORKDIR}/data/sft/{name}"
    path = base if os.path.exists(base) else base + ".gz"
    if not os.path.exists(path):
        n = 0
    elif path.endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8") as fh:
            n = sum(1 for _ in fh)
    else:
        n = sum(1 for _ in open(path, encoding="utf-8"))
    print(name, "rows=", n, "(", os.path.basename(path), ")")
    assert n > 0, f"missing/empty {path} - commit the regenerated corpus to the branch first"


In [ ]:
# Cell 6.6 — PROOF GATE (FREE, ~25 min, single GPU). Run BEFORE the 11h train.
# Hard-overfits 256 rows with the new name-weighted loss, then free-gen probes them. v3 fixed
# the FORMAT; this gate proves the NEW thing: the skill/tool NAME is now exact, not garbled.
#   PASS: OK>=5 AND names-match-gold>=4 -> run Cell 7 (full train).
#   FAIL: stop, tell Opus; do not burn the session.
import subprocess, sys, os, json, gzip, re
MICRO = "micro_overfit"
cmd = [sys.executable, "-m", "llm_training.run_train",
       "--model", MODEL, "--engine", ENGINE, "--max-seq", str(MAX_SEQ),
       "--rank", str(RANK), "--targets", "all-linear", "--grad-accum", "1",
       "--max-steps", "400", "--max-examples", "256", "--max-val", "0",
       "--lr", "2e-4", "--data-stem", DATA_STEM, "--output", MICRO]
print(">", " ".join(cmd), flush=True)
r = subprocess.run(cmd, cwd=WORKDIR, env={**os.environ, "PYTHONPATH": f"{WORKDIR}/src/llm",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        "TORCHDYNAMO_DISABLE": "1", "UNSLOTH_COMPILE_DISABLE": "1"})
assert r.returncode == 0, "micro-overfit FAILED — read the trainer output above"

sys.path.insert(0, f"{WORKDIR}/src/llm")
os.environ["CHESS_HF_BASE"] = f"{WORKDIR}/src/llm/models/{MODEL}"
os.environ["CHESS_REP_PENALTY"] = "1.0"; os.environ["CHESS_NO_REPEAT_NGRAM"] = "0"  # clean decode
from backend.model_hf import HFModel
from llm_training.system_prompt import build_system
model = HFModel(adapter=f"{WORKDIR}/runs/{MICRO}", temperature=0.0)

rows = []
p = f"{WORKDIR}/data/sft/{DATA_STEM}_train.jsonl"
if not os.path.exists(p): p += ".gz"
op = gzip.open if p.endswith(".gz") else open
with op(p, "rt", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 256: break
        line = line.strip()
        if line: rows.append(json.loads(line))
def pick(m, n): return [x for x in rows if x.get("reasoning_mode") == m][:n]
samples = pick("fast", 3) + pick("think", 3) + pick("auto", 2)

CONTRACT = {"think","/think","goal","/goal","plan","/plan","skill","/skill","tool","/tool"}
def first_name(text):
    m = re.search(r"<skill>\s*([\w./-]+)", text) or re.search(r"<tool>\s*([\w./-]+)", text)
    return m.group(1) if m else None
def classify(out):
    foreign = sorted({t for t in re.findall(r"</?([a-zA-Z_][\w]*)", out) if t not in CONTRACT})
    emits = ("<skill>" in out) or ("<tool>" in out)
    return ("OK" if emits and not foreign else "SOUP" if foreign else "NO-EMIT")

tally = {"OK": 0, "SOUP": 0, "NO-EMIT": 0}; name_match = 0
for x in samples:
    sysp = build_system(x["skills_index"], x["tool_manifest"], x.get("plugin_context", {}),
                        reasoning_mode=x.get("reasoning_mode", ""))
    user = next(m["content"] for m in x["messages"] if m["role"] == "user")
    gold = next(m["content"] for m in x["messages"] if m["role"] == "assistant")
    out = model.generate([{"role": "system", "content": sysp}, {"role": "user", "content": user}],
                         max_new_tokens=120, stop=["</tool>", "</skill>"])
    v = classify(out); tally[v] += 1
    gname, mname = first_name(gold), first_name(out)
    ok_name = gname is not None and gname == mname
    name_match += int(ok_name)
    print("=" * 64); print(f"[{x.get('reasoning_mode')}] {v}  name_gold={gname} name_model={mname} {'MATCH' if ok_name else ''}")
    print("USER :", user[:80]); print("MODEL:", out[:200])
print(f"\nGATE: format {tally}  | names matched gold {name_match}/{len(samples)}")
print("=> PASS — run Cell 7" if tally["OK"] >= 5 and name_match >= 4
      else "=> FAIL — stop, tell Opus; do not run the full session")


In [ ]:
# Cell 7 - TRAIN (E4B QLoRA, Unsloth, all-linear, format-weighted loss). The ONLY cell you
# wait on. DDP=True -> accelerate launch 2 procs (~2x). Checkpoints every SAVE_EVERY + on
# timeout/crash; best/ = lowest val (-> SERVE). With CKPT_REPO set, best/ + checkpoint/ mirror
# to the Hub each save, so a 12h kill can't lose progress and a re-run resumes automatically.
import subprocess, sys, os
args = ['--model', MODEL, '--engine', ENGINE, '--max-steps', str(MAX_STEPS),
        '--rank', str(RANK), '--targets', TARGETS, '--grad-accum', str(GRAD_ACCUM),
        '--lr', str(LR), '--max-seq', str(MAX_SEQ), '--eval-every', str(EVAL_EVERY),
        '--max-val', str(MAX_VAL), '--save-every', str(SAVE_EVERY),
        '--data-stem', DATA_STEM, '--output', OUTPUT]
if NO_4BIT: args.append('--no-4bit')
if RESUME:  args.append('--resume')
if DDP:
    cmd = ['accelerate','launch','--num_processes','2','--multi_gpu','-m','llm_training.run_train'] + args
else:
    cmd = [sys.executable,'-m','llm_training.run_train'] + args
print('>', ' '.join(cmd))
subprocess.run(cmd, check=True, cwd=WORKDIR,
               env={**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm',
                    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'})


In [ ]:
# Cell 8 — package adapter for download (OPTIONAL; run after training OR a timeout).
# Resume is now AUTOMATIC via the Hub (Cell 4.5 pulls+confirms before weight loading), so
# you do NOT need this zip to continue — it's just a convenient local copy of the
# final/best adapter to pull onto the 4060. runs/OUTPUT has: best/ (lowest-val -> SERVE) +
# checkpoint/ (latest, for resume) + the final adapter. With CKPT_REPO set, best/ and
# checkpoint/ are already on the Hub.
import shutil, os
run_dir = f"{WORKDIR}/runs/{OUTPUT}"
print("run dir contents:", os.listdir(run_dir) if os.path.isdir(run_dir) else "MISSING")
out_zip = f"/kaggle/working/{OUTPUT}"
shutil.make_archive(out_zip, "zip", run_dir)
sz = os.path.getsize(out_zip + ".zip") / 1e6
print(f"\n✅ {out_zip}.zip ({sz:.1f} MB) — download from the Output panel (optional).")
print("   SERVE  -> the best/ folder inside (lowest val), or pull best/ from CKPT_REPO.")
print("   RESUME -> AUTOMATIC next session: just re-run the notebook (Cell 4.5 pulls the")
print("             latest checkpoint from CKPT_REPO and confirms it). No upload needed.")
